# LC 1091 — Shortest Path in Binary Matrix
**Difficulty:** Medium | **Pattern:** BFS on Grid (8-directional)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> BFS on an unweighted grid always
finds the shortest path. The twist here is 8-directional
movement (diagonals included) and cells must be 0 to traverse.
Mark cells visited as soon as they are enqueued — not when
dequeued — to avoid redundant work.
</div>

## Official Problem Statement

Given an `n x n` binary matrix `grid`, return the length of the
**shortest clear path** from the top-left `(0,0)` to the
bottom-right `(n-1,n-1)`. If there is no clear path, return
`-1`.

A **clear path** is a path from `(0,0)` to `(n-1,n-1)` such
that all visited cells are `0`, and 8-directionally adjacent.
The **length** of a clear path is the number of visited cells.

**Constraints:**
- `n == grid.length == grid[i].length`
- `1 <= n <= 100`
- `grid[i][j]` is `0` or `1`

## What This Is Actually Asking

Find the fewest cells you must step on to travel from corner to
corner on a grid where `1` cells are walls. You can move in all
8 directions (including diagonals), but only through `0` cells.
The path length counts every cell stepped on, including start
and end. BFS guarantees the first time you reach the destination
is the shortest; anything else would be slower.

## Walk Through an Example by Hand

```
grid = [[0,0,0],
        [1,1,0],
        [1,1,0]]

Start=(0,0), End=(2,2)

BFS:
  Step 1: enqueue (0,0) dist=1
  Step 2: from (0,0) → (0,1) dist=2
  Step 3: from (0,1) → (0,2),(1,2) dist=3
  Step 4: from (0,2),(1,2) → (2,2) dist=4  ← DONE

Answer = 4
```

If start or end is `1`, return -1 immediately.

## The Picture

```
grid (0=clear, 1=wall):
  [0, 0, 0]
  [1, 1, 0]
  [1, 1, 0]

8 directions:
  (-1,-1)(-1,0)(-1,+1)
   (0,-1)  [X]  (0,+1)
  (+1,-1)(+1,0)(+1,+1)

BFS wave expansion (. = visited, d = dist):
  d=1     d=2     d=3     d=4
  S . .   S 2 .   S 2 3   S 2 3
  # # .   # # .   # # 3   # # 3
  # # .   # # .   # # .   # # 4←

Key:
  - Mark cell visited when ENQUEUED (grid[r][c]=1 or use set)
  - Store distance alongside coordinates: (r, c, dist)
  - Check bounds and grid[r][c]==0 before enqueueing
```

## When To Use This Pattern

- When asked for the **shortest path** on an unweighted grid,
  think **BFS** (not DFS, not Dijkstra).
- When movement is **8-directional**, expand the directions
  tuple to all 8 combinations of (-1, 0, +1) × (-1, 0, +1)
  minus (0, 0).
- When the start or end cell is blocked (`1`), think **early
  exit returning -1 before BFS**.
- When asked for path **length** (not just reachability), think
  **track distance inside queue tuples**.
- When n ≤ 100, BFS O(n²) is trivially fast.

## The Approach

Guard against blocked start/end cells first. Enqueue `(0, 0)`
with distance 1, and mark it visited immediately. Process the
queue: for each cell, try all 8 neighbours; if in bounds and
unvisited and equal to 0, enqueue with distance+1 and mark
visited. If you dequeue `(n-1, n-1)`, return its distance.
If the queue empties without reaching the goal, return -1.

In [ ]:
from typing import List
from collections import deque

In [ ]:
def test_harness(func):
    cases = [
        # (grid, expected)
        ([[0,1],[1,0]],                      2),
        ([[0,0,0],[1,1,0],[1,1,0]],           4),
        ([[1,0,0],[1,1,0],[1,1,0]],          -1),  # start blocked
        ([[0]],                               1),  # 1x1
        ([[1]],                              -1),  # 1x1 blocked
        ([[0,0],[0,0]],                       2),  # 2x2 clear
        ([[0,1],[0,0]],                       3),  # must go around
    ]
    passed = 0
    for grid, expected in cases:
        import copy
        result = func(copy.deepcopy(grid))
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: grid={grid} "
                f"=> got {result}, want {expected}"
            )
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
def shortest_path_binary_matrix(grid: List[List[int]]) -> int:
    """
    Return shortest clear path length from (0,0) to (n-1,n-1).

    Strategy: BFS with 8-directional expansion.
      - Guard: if grid[0][0]==1 or grid[n-1][n-1]==1 → -1
      - Enqueue (0,0,1); mark visited.
      - Expand in 8 directions; track distance.
      - Return dist when (n-1,n-1) reached, else -1.

    Args:
        grid: n x n binary grid (0=clear, 1=wall)
    Returns:
        Length of shortest clear path, or -1.
    """
    # Debug: confirm grid size and corners
    # n = len(grid)
    # print(f"n={n}, start={grid[0][0]}, end={grid[n-1][n-1]}")

    pass

    # Debug: print distance each time goal is close
    # print(f"Reached ({n-1},{n-1}) at dist={dist}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(shortest_path_binary_matrix)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| DFS (finds A path) | O(n²) | O(n²) | Not guaranteed shortest |
| Dijkstra | O(n² log n) | O(n²) | Overkill for unit weights |
| **BFS** | **O(n²)** | **O(n²)** | Optimal for unweighted |

## Real World Connection

At **Citi**, finding the shortest compliant trade route through
a network of approved counterparties (where blocked nodes are
sanctioned entities) uses this exact BFS pattern. **AWS**
network routing through VPC subnets — where some subnets are
in blackout maintenance — follows the same 8-connected-grid
analogy. For a **data engineer**, navigating a data lineage
graph to find the shortest transformation path from raw
source to a clean table mirrors BFS on a binary obstacle grid.
The 8-directional variant models systems where diagonal
(indirect) connections also exist.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra